# **TensorBoard · MLflow · Weights & Biases · обзор Comet · сравнение платформ**

## Проверка окружения

In [5]:
import sys, shutil
import torch, torchvision, tensorboard, mlflow, wandb, sklearn, pandas

print("ok, torch", torch.__version__)
print("python:     ", sys.executable)
print("tensorboard:", shutil.which("tensorboard"))

ok, torch 2.15.0+cpu
python:      /home/defenser/Documents/Mipt-projects/mipt-seminars/.venv/bin/python
tensorboard: /home/defenser/Documents/Mipt-projects/mipt-seminars/.venv/bin/tensorboard


In [6]:
from cleanup_tracking import clean_all

clean_all(remote=True)
# clean_all(dry_run=True)    # только показать, что будет удалено
# clean_all(remote=True)     # + удалить запуски в облаке W&B и Comet (необратимо)

Очистка артефактов в /home/defenser/Documents/Mipt-projects/mipt-seminars/tracking-seminar
• MLflow: остановка сервера
  остановлен сервер MLflow, pid: [20380, 20435, 20436, 20479, 20480, 20483, 20484, 20487, 20489, 20490]
• Локальные файлы (TensorBoard, MLflow, W&B, Comet)
  ./data сохранён (include_data=True — удалить)
  удалено: runs
  удалено: mlflow.db
  удалено: mlflow_server.log
• Облако: W&B (необратимо)
  W&B недоступен или проекта нет (AuthenticationError: Failed to verify credentials with https://api.forge.coreweave.com: Failed to execute API request: the service process is busy and did not respond in time.)
• Облако: Comet (необратимо)
  в Comet проекте defenser1337/seminar-tracking экспериментов нет
Готово.


{'removed': ['runs', 'mlflow.db', 'mlflow_server.log'],
 'mlflow_pids': [20380,
  20435,
  20436,
  20479,
  20480,
  20483,
  20484,
  20487,
  20489,
  20490],
 'wandb_runs': 0,
 'comet_experiments': 0}

## Оглавление
0. [Сквозной бейзлайн](#baseline) — один эксперимент для всех платформ
1. [Мотивация: зачем нужно отслеживание](#sec0) — что отслеживать, примитивный логгер
2. [TensorBoard](#sec1) — SummaryWriter, скаляры, гистограммы, изображения, проектор, системные метрики, ограничения
3. [MLflow](#sec2) — свой сервер, трекинг, автолог, системные метрики, реестр моделей
4. [Weights & Biases](#sec3) — config/log, артефакты, sweeps, отчёты, офлайн
5. [Comet](#sec4) — адаптер, confusion matrix, артефакты, offline-режим, ограничения
6. [Сравнение платформ](#sec5) — таблица и другие инструменты
7. [Заключение](#sec6) — выводы, чек-лист, ИИ-агенты, кейсы для дискуссии
8. [Источники и ссылки](#appendix) — документация и материалы

In [7]:
import os, subprocess, time, random, json, pathlib, warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

torch 2.15.0+cpu | device: cpu


<a id="baseline"></a>
# 0. Сквозной бейзлайн: один эксперимент — все платформы

Чтобы честно сравнить инструменты, **один и тот же эксперимент** мы залогируем в TensorBoard, MLflow и W&B.

Задача: классификация **FashionMNIST** маленькой CNN.

Каждая платформа получит свой **адаптер-логгер** с одним и тем же интерфейсом:

| метод адаптера | что делает |
|---|---|
| `log_params(cfg)` | сохранить гиперпараметры |
| `log_metric(name, value, step)` | метрика по шагам/эпохам |
| `log_image(...)` / `log_artifact(...)` | картинки и файлы (есть не у всех) |
| `finish()` | закрыть запуск |


In [8]:
DATA_DIR = "./data"
MEAN, STD = 0.2860, 0.3530 

TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])

train_full = torchvision.datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=TRANSFORM)
test_set   = torchvision.datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=TRANSFORM)
vis_set    = torchvision.datasets.FashionMNIST(DATA_DIR, train=False, download=True,
                                               transform=transforms.ToTensor())

CLASS_NAMES = list(train_full.classes)
N_TRAIN, N_VAL = 8000, 2000       
train_set = Subset(train_full, range(N_TRAIN))
val_set   = Subset(train_full, range(N_TRAIN, N_TRAIN + N_VAL))
print("классы:", CLASS_NAMES)


def make_loaders(batch_size):
    return {
        "train": DataLoader(train_set, batch_size=batch_size, shuffle=True,
                            num_workers=2, drop_last=True),
        "val":   DataLoader(val_set, batch_size=256, num_workers=2),
    }


def normalize_batch(x):
    return (x - MEAN) / STD

классы: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


In [9]:
class SmallCNN(nn.Module):
    def __init__(self, hidden_dim=64, dropout=0.2):
        super().__init__()
        c1, c2 = max(8, hidden_dim // 4), max(16, hidden_dim // 2)
        self.features = nn.Sequential(
            nn.Conv2d(1, c1, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(c1, c2, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(c2 * 7 * 7, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

    def embed(self, x):
        """Скрытый вектор (последний Linear выкидываем) — для проектора эмбеддингов."""
        return self.classifier[:-1](self.features(x))


def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            total_loss += F.cross_entropy(logits, y, reduction="sum").item()
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total


CFG = {"lr": 1e-3, "batch_size": 128, "hidden_dim": 64, "dropout": 0.2, "epochs": 3}


def train_model(cfg, logger=None, verbose=True):
    """Сквозной бейзлайн доклада.

    logger — любой адаптер с интерфейсом:
      log_params(cfg) | log_metric(name, value, step=) | finish()
    """
    torch.manual_seed(SEED)
    loaders = make_loaders(cfg["batch_size"])
    model = SmallCNN(cfg["hidden_dim"], cfg["dropout"]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])

    if logger is not None:
        logger.log_params(cfg)

    step = 0
    for epoch in range(cfg["epochs"]):
        model.train()
        for x, y in loaders["train"]:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            opt.step()
            step += 1
            if logger is not None:
                logger.log_metric("train/loss", loss.item(), step)
        val_loss, val_acc = evaluate(model, loaders["val"])
        if logger is not None:
            # тот же global step, что и у train/*: иначе val-точки уедут к нулю на общей оси X
            logger.log_metric("val/loss", val_loss, step)
            logger.log_metric("val/acc", val_acc, step)
        if verbose:
            print(f"эпоха {epoch}: val_loss={val_loss:.4f}  val_acc={val_acc:.3f}")

    if logger is not None:
        logger.finish()
    return model

In [10]:
model = train_model(CFG, logger=None)

эпоха 0: val_loss=0.6747  val_acc=0.745
эпоха 1: val_loss=0.5281  val_acc=0.794
эпоха 2: val_loss=0.4796  val_acc=0.828


<a id="sec0"></a>
# 1. Мотивация: необходимость систематического отслеживания экспериментов

Типичная практика ведения экспериментов в машинном обучении часто выглядит так:

- результаты, распределённые по десяткам CSV-файлов;
- графики, сохранённые в переписке или в виде скриншотов;
- невозможность воспроизвести лучший запуск ни самому автору, ни коллегам.


<figure style="text-align: center; margin: 1.5em;">
  <img src="images_seminar_tracking/problems_without_experiment_tracking.png" alt="Проблемы ведения экспериментов без трекинга" style="width: 85%; max-width: 900px;">
  <figcaption style="font-size: 0.9em; color: gray; margin-top: 0.5em;">
    Рис. 1. Типичные проблемы при ведении экспериментов без систематического отслеживания
  </figcaption>
</figure>

---

## Объекты отслеживания

| Объект | Обоснование | Пример |
|---|---|---|
| **Динамика метрик** | Переобучение и недообучение видны в ходе обучения, а не по его завершении | `train_loss`, `val_loss` по эпохам |
| **Гиперпараметры** | Без зафиксированной конфигурации результат невоспроизводим | `lr=3e-4`, `batch_size=64`, `seed=42` |
| **Артефакты** | Чекпоинты, графики и версии датасетов нужны для анализа и повторного использования | `best_model.pt`, `confusion_matrix.png` |
| **Системные показатели** | Позволяют диагностировать нехватку памяти (OOM) и узкие места, оценить стоимость эксперимента | загрузка GPU 40%, пик памяти 23 ГБ |
| **Сравнение запусков** | Десятки запусков анализируются в одной таблице или на одном графике | «запуск → метрики → параметры» |
| **Совместная работа** | Руководитель и команда могут обсуждать и повторять эксперименты | общая ссылка на дашборд |

---

## Демонстрация: примитивный логгер

Рассмотрим минимальную реализацию трекинга без специализированных инструментов: для каждого запуска сохраняется конфигурация (`config.json`), а финальные метрики дописываются в общий файл `results.csv`.

Даже такой аккуратный подход имеет ограничения: фиксируются только итоговые значения метрик, не сохраняются версии кода и окружения, формат таблицы жёстко задан первым запуском, а сравнение запусков выполняется вручную. Именно эти задачи решают платформы трекинга.

In [7]:
import csv, datetime, json, pathlib, shutil
import pandas as pd

RUNS_DIR = pathlib.Path("runs/naive")
shutil.rmtree(RUNS_DIR, ignore_errors=True) 
RUNS_DIR.mkdir(parents=True, exist_ok=True)


def naive_log_csv(cfg, metrics):
    """Конфиг запуска -> отдельная папка; финальные метрики -> общий results.csv."""
    stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    run_dir = RUNS_DIR / f"{stamp}_lr{cfg['lr']}"
    run_dir.mkdir(parents=True, exist_ok=True)

    # 1) конфигурация: runs/naive/<время>_lr<...>/config.json
    with open(run_dir / "config.json", "w") as f:
        json.dump(cfg, f, indent=2)

    # 2) метрики: runs/naive/results.csv (одна строка = один запуск)
    results = RUNS_DIR / "results.csv"
    is_new = not results.exists()
    with open(results, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["run_dir", *metrics])
        if is_new:
            writer.writeheader()
        writer.writerow({"run_dir": str(run_dir), **metrics})

    return run_dir


# Три «запуска» с разным lr (значения метрик условные)
demo_runs = [
    (1e-2, {"val_acc": 0.81, "val_loss": 0.52}),
    (1e-3, {"val_acc": 0.87, "val_loss": 0.36}),
    (3e-4, {"val_acc": 0.85, "val_loss": 0.41}),
]
for lr, metrics in demo_runs:
    naive_log_csv({**CFG, "lr": lr}, metrics)


# Что получилось на диске
print("Структура папок:")
for p in sorted(RUNS_DIR.rglob("*")):
    depth = len(p.relative_to(RUNS_DIR).parts) - 1
    print("  " * depth + p.name + ("/" if p.is_dir() else ""))

# Сводная таблица результатов
df = pd.read_csv(RUNS_DIR / "results.csv")
df.sort_values("val_acc", ascending=False).reset_index(drop=True)

Структура папок:
2026-11-03_17-24-52_lr0.0003/
  config.json
2026-11-03_17-24-52_lr0.001/
  config.json
2026-11-03_17-24-52_lr0.01/
  config.json
results.csv


,run_dir,val_acc,val_loss
0,runs/naive/2026-11-03_17-24-52_lr0.001,0.87,0.36
1,runs/naive/2026-11-03_17-24-52_lr0.0003,0.85,0.41
2,runs/naive/2026-11-03_17-24-52_lr0.01,0.81,0.52


**Что этот «логгер» потерял по сравнению со специализированными инструментами:**

<figure style="text-align: center; margin: 0.5em;">
  <img src="images_seminar_tracking/naive_logger_gaps.png" alt="Что теряет примитивный логгер" style="width: 85%; max-width: 900px;">
  <figcaption style="font-size: 0.9em; color: gray;">
    Рис. 2. Возможности, недоступные при ручном ведении логов
  </figcaption>
</figure>

<a id="sec1"></a>
# 2. TensorBoard

**Что здесь покажем:** локальный запуск UI, `SummaryWriter` (скаляры, гистограммы, изображения, проектор эмбеддингов) и честный список ограничений.

TensorBoard хранит логи **файлами** (`events.out.tfevents.*`) в заданном каталоге — UI их только читает.

> Из терминала: `tensorboard --logdir runs/tb`.

In [8]:
class TensorBoardLogger:
    """Адаптер: интерфейс бейзлайна -> SummaryWriter."""

    def __init__(self, logdir):
        self.writer = SummaryWriter(log_dir=logdir) 

    def log_params(self, cfg):
        # У TensorBoard нет отдельного «пространства параметров» — пишем текстом
        for k, v in cfg.items():
            self.writer.add_text(f"params/{k}", str(v))

    def log_metric(self, name, value, step=0):
        self.writer.add_scalar(name, value, global_step=step)

    def log_histogram(self, name, tensor, step=0):
        self.writer.add_histogram(name, tensor, global_step=step)

    def log_embedding(self, feats, labels, images=None):
        self.writer.add_embedding(feats, metadata=labels, label_img=images)

    def finish(self):
        self.writer.flush()
        self.writer.close()


# Чистим прошлые прогоны: иначе повторный запуск ячейки добавит второй events-файл
# в тот же каталог, и TensorBoard сольёт старые и новые точки в одну кривую.
shutil.rmtree("runs/tb", ignore_errors=True)

# Два запуска с разными lr: TensorBoard сам положит их рядом (по подкаталогам)
for lr in (1e-3, 1e-4):
    logger = TensorBoardLogger(f"runs/tb/lr={lr}")
    train_model({**CFG, "lr": lr}, logger=logger)

print("В терминале: tensorboard --logdir runs/tb → вкладка SCALARS: сравните train/loss и val/acc")

эпоха 0: val_loss=0.6747  val_acc=0.745
эпоха 1: val_loss=0.5281  val_acc=0.794
эпоха 2: val_loss=0.4796  val_acc=0.828
эпоха 0: val_loss=1.6346  val_acc=0.568
эпоха 1: val_loss=0.9891  val_acc=0.704
эпоха 2: val_loss=0.8014  val_acc=0.723
В терминале: tensorboard --logdir runs/tb → вкладка SCALARS: сравните train/loss и val/acc


In [9]:
# Изображения: примеры из валидации + предсказания модели (правильные зелёным, ошибки красным)
xs, ys = next(iter(DataLoader(vis_set, batch_size=16, shuffle=True)))
model.eval()
with torch.no_grad():
    preds = model(normalize_batch(xs).to(DEVICE)).argmax(1).cpu()

fig = plt.figure(figsize=(12, 5))
for i in range(16):
    plt.subplot(2, 8, i + 1)
    plt.imshow(xs[i, 0], cmap="gray")
    plt.title(CLASS_NAMES[preds[i]], fontsize=8,
              color="green" if preds[i] == ys[i] else "red")
    plt.axis("off")

# model здесь — бейзлайн из секции 0 (тот же CFG и seed, что у запуска lr=1e-3).
# Пишем в каталог существующего запуска: второй events-файл в том же каталоге
# TensorBoard объединяет в ОДНУ карточку запуска — картинки не отрываются от run.
img_writer = SummaryWriter("runs/tb/lr=0.001")
img_writer.add_figure("val/predictions", fig)
img_writer.close()
plt.close(fig)
print("TensorBoard → запуск lr=0.001 → вкладка IMAGES")

TensorBoard → запуск lr=0.001 → вкладка IMAGES


In [10]:
# Проектор эмбеддингов: как выглядит пространство скрытых представлений модели?
model = train_model(CFG, logger=None, verbose=False) 

emb_writer = SummaryWriter("runs/tb/lr=0.001/embeddings")
emb_loader = DataLoader(Subset(vis_set, range(1000)), batch_size=250)

feats, labels, imgs = [], [], []
model.eval()
with torch.no_grad():
    for x, y in emb_loader:
        # нормализация та же, что при обучении: иначе модель видит другое распределение
        feats.append(model.embed(normalize_batch(x).to(DEVICE)).cpu())
        labels.extend(CLASS_NAMES[i] for i in y.tolist())
        imgs.append(x)

feats = torch.cat(feats)
imgs = torch.cat(imgs)
emb_writer.add_embedding(feats, metadata=labels, label_img=imgs)
emb_writer.close()
print("TensorBoard → PROJECTOR → run lr=0.001/embeddings: классы должны кластероваться")

TensorBoard → PROJECTOR → run lr=0.001/embeddings: классы должны кластероваться


## Системные показатели и метрики GPU

В TensorBoard **встроенного мониторинга нет**, и отдельной вкладки «System», как в MLflow/W&B, тоже нет —
всё, что вы напишете, попадает в обычные скаляры. Варианты обхода:
- логировать `psutil` (CPU/RAM) и `nvidia-smi` (GPU) скалярами вручную — работает и на машине без GPU;
- `torch.profiler` — профилирование производительности (трейсы, а не мониторинг за время обучения).

In [11]:
import subprocess

try:
    import psutil      
except ImportError:
    psutil = None


def system_stats():
    """CPU/RAM/память процесса — всегда. GPU — только если есть nvidia-smi.

    Без GPU раньше возвращался None и в лог не попадало ничего; теперь базовые
    метрики есть на любой машине, а GPU-ключи просто не появляются.
    """
    stats = {}
    if psutil is not None:
        stats["cpu/util_percent"] = psutil.cpu_percent(interval=0.1)
        vm = psutil.virtual_memory()
        stats["ram/used_mib"] = vm.used / 2**20
        stats["ram/percent"] = vm.percent
        stats["proc/rss_mib"] = psutil.Process().memory_info().rss / 2**20
    else:
        stats["cpu/loadavg_1m"] = os.getloadavg()[0]

    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
             "--format=csv,noheader,nounits"], text=True)
        util, mem = [float(v) for v in out.strip().splitlines()[0].split(",")]
        stats["gpu/util_percent"] = util
        stats["gpu/mem_mib"] = mem
    except Exception:
        pass  
    return stats


sys_writer = SummaryWriter("runs/tb/system")
for step in range(5):
    stats = system_stats()
    for name, value in stats.items():
        sys_writer.add_scalar(name, value, global_step=step)
    time.sleep(0.5)
sys_writer.close()

print("залогировано:", ", ".join(sorted(stats)))

залогировано: cpu/util_percent, proc/rss_mib, ram/percent, ram/used_mib


In [ ]:
# Демонстрация: гистограммы весов и градиентов — как распределения «живут» по шагам
ex_writer = SummaryWriter("runs/tb/histograms")
model_ex = SmallCNN().to(DEVICE)
opt_ex = torch.optim.Adam(model_ex.parameters(), lr=1e-3)
loader_ex = make_loaders(CFG["batch_size"])["train"]

for step, (x, y) in enumerate(loader_ex):
    x, y = x.to(DEVICE), y.to(DEVICE)
    opt_ex.zero_grad()
    loss = F.cross_entropy(model_ex(x), y)
    loss.backward()
    opt_ex.step()

    fc1 = model_ex.classifier[1]                       # первый nn.Linear
    ex_writer.add_histogram("weights/fc1", fc1.weight, global_step=step)
    ex_writer.add_histogram("grads/fc1", fc1.weight.grad, global_step=step)
    ex_writer.add_scalar("train/loss", loss.item(), step)
    if step >= 60:
        break

ex_writer.close()
print("Откройте вкладку HISTOGRAMS: как распределения меняются по шагам?")
print("Гистограммы градиентов — быстрый способ поймать «взрыв/затухание градиента».")

## Ограничения TensorBoard

<figure style="text-align: center; margin: 0.5em;">
  <img src="images_seminar_tracking/tensorboard_limitations.png"
       alt="Основные ограничения TensorBoard"
       style="width: 85%; max-width: 900px;">
  <figcaption style="font-size: 0.9em; color: gray;">
    Рис. 3. Ограничения TensorBoard: совместная работа, сравнение запусков,
    управление артефактами и программный доступ к результатам
  </figcaption>
</figure>

<a id="sec2"></a>
# 3. MLflow

Open-source платформа от LF AI & Data. Главные фишки:
- **самостоятельный хостинг** — все данные (метрики, артефакты, модели) лежат у вас (SQLite/Postgres + S3/диск);
- ручной трекинг + **автологирование** для десятков библиотек;
- сравнение запусков в UI и из кода (`search_runs` → pandas);
- **реестр моделей** (в MLflow 3 модели — сущности первого класса, алиасы `@champion`).

## Концепции
**Experiment** (проект) → **Run** (запуск) → внутри: `params` (гиперпараметры), `metrics` (по шагам), `tags`, `artifacts` (файлы), `models`.

In [12]:
import os, subprocess, sys, time, urllib.request

# Системные метрики: клиент поднимает фоновый монитор на каждый run.
# Переменные читаются в момент start_run(), поэтому ставим их до первых запусков.
os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"
os.environ["MLFLOW_SYSTEM_METRICS_SAMPLING_INTERVAL"] = "2"   # по умолчанию 10 с — для демо редко

MLFLOW_PORT = 5000
MLFLOW_URI = f"http://127.0.0.1:{MLFLOW_PORT}"


def server_is_up():
    try:
        return urllib.request.urlopen(f"{MLFLOW_URI}/health", timeout=2).status == 200
    except Exception:
        return False


mlflow_server = None   # если сервер уже был поднят кем-то ещё, гасить его в конце не будем
if server_is_up():
    print(f"Сервер уже работает на порту {MLFLOW_PORT} — переиспользуем его")
else:
    server_log = open("mlflow_server.log", "wb")
    mlflow_server = subprocess.Popen(
        [sys.executable, "-m", "mlflow", "server",
         "--host", "127.0.0.1", "--port", str(MLFLOW_PORT),
         "--workers", "1",
         "--backend-store-uri", "sqlite:///mlflow.db",
         "--default-artifact-root", "./mlartifacts"],
        stdout=server_log, stderr=server_log,
    )
    for _ in range(30):                    # до ~30 с: первый запуск делает миграции БД
        if server_is_up():
            break
        if mlflow_server.poll() is not None:
            tail = open("mlflow_server.log", "rb").read().decode(errors="replace")[-1500:]
            raise RuntimeError(f"MLflow server упал. Хвост лога:\n{tail}")
        time.sleep(1)
    else:
        raise RuntimeError("MLflow server не ответил за 30 секунд — см. mlflow_server.log")

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment("seminar-tracking")
print("MLflow UI →", MLFLOW_URI, "(откройте в новой вкладке!)")

2026/11/03 17:47:56 INFO mlflow.tracking.fluent: Experiment with name 'seminar-tracking' does not exist. Creating a new experiment.


MLflow UI → http://127.0.0.1:5000 (откройте в новой вкладке!)


In [13]:
class MLflowLogger:
    """Адаптер: интерфейс бейзлайна -> MLflow."""

    def __init__(self, run_name=None):
        self.run = mlflow.start_run(run_name=run_name)
        self.run_id = self.run.info.run_id   # нужен, чтобы дописать артефакт после finish()

    def log_params(self, cfg):
        mlflow.log_params(cfg)

    def log_metric(self, name, value, step=0):
        mlflow.log_metric(name, value, step=step)

    def log_artifact(self, path):
        mlflow.log_artifact(path)

    def log_dict(self, obj, filename):
        mlflow.log_dict(obj, filename)

    def finish(self):
        mlflow.end_run()


# Пять конфигураций — мини-свип по lr и ширине скрытого слоя.
# Именно из них потом выберем лучшую и отправим в реестр моделей.
EXPERIMENTS = [
    {"lr": 1e-4, "hidden_dim": 32},
    {"lr": 3e-4, "hidden_dim": 64},
    {"lr": 1e-3, "hidden_dim": 64},
    {"lr": 1e-3, "hidden_dim": 128},
    {"lr": 3e-3, "hidden_dim": 64},
]

results = []   # run_id, model_uri и val/acc каждого запуска
for params in EXPERIMENTS:
    cfg = {**CFG, **params}
    logger = MLflowLogger(run_name=f"cnn_lr={cfg['lr']}_h={cfg['hidden_dim']}")
    model = train_model(cfg, logger=logger, verbose=False)

    # train_model закрыл run (logger.finish) — возобновляем его по run_id,
    # чтобы модель стала артефактом именно этого запуска, а не нового пустого.
    with mlflow.start_run(run_id=logger.run_id):
        info = mlflow.pytorch.log_model(
            model.to("cpu"), name="model",
            input_example=torch.randn(1, 1, 28, 28),   # задаёт подпись входа
            # pt2 требует TensorSpec и не даёт pyfunc; для демо берём pickle.
            # MLflow предупредит, что pickle небезопасен при загрузке, — это ожидаемо.
            serialization_format="pickle",
        )

    results.append({
        "run_id": logger.run_id,
        "model_uri": info.model_uri,                  # models:/m-...
        "val_acc": mlflow.get_run(logger.run_id).data.metrics["val/acc"],
    })

print("В UI: эксперимент seminar-tracking → таблица запусков → колонка val/acc")


2026/11/03 17:47:59 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/11/03 17:47:59 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/11/03 17:48:05 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/11/03 17:48:05 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
2026/11/03 17:48:05 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/11/03 17:48:05 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


🏃 View run cnn_lr=0.0001_h=32 at: http://127.0.0.1:5000/#/experiments/1/runs/6f88a934e985467cae55877637bd0186
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/11/03 17:48:05 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/11/03 17:48:05 WARNING mlflow.utils.requirements_utils: Found torch version (2.15.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.15.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/03 17:48:09 WARNING mlflow.utils.requirements_utils: Found torch version (2.15.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.15.0' without the local version label to m

🏃 View run cnn_lr=0.0001_h=32 at: http://127.0.0.1:5000/#/experiments/1/runs/6f88a934e985467cae55877637bd0186
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/11/03 17:48:17 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/11/03 17:48:17 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
2026/11/03 17:48:17 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/11/03 17:48:17 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


🏃 View run cnn_lr=0.0003_h=64 at: http://127.0.0.1:5000/#/experiments/1/runs/affe4f422d314d50b201c7037aa7354a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


... (10 outputs left out)

2026/11/03 17:49:04 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/11/03 17:49:04 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
2026/11/03 17:49:04 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/11/03 17:49:04 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


🏃 View run cnn_lr=0.003_h=64 at: http://127.0.0.1:5000/#/experiments/1/runs/ed6803499aa141bcb38da8a6723053c2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/11/03 17:49:04 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/11/03 17:49:04 WARNING mlflow.utils.requirements_utils: Found torch version (2.15.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.15.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/03 17:49:08 WARNING mlflow.utils.requirements_utils: Found torch version (2.15.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.15.0' without the local version label to m

🏃 View run cnn_lr=0.003_h=64 at: http://127.0.0.1:5000/#/experiments/1/runs/ed6803499aa141bcb38da8a6723053c2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
В UI: эксперимент seminar-tracking → таблица запусков → колонка val/acc


## Автологирование

Один вызов — и параметры, метрики, модель сохраняются сами. Проверим на sklearn (быстро и надёжно),
а для PyTorch есть `mlflow.pytorch.autolog()`.

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train = train_full.data[:5000].reshape(-1, 784).numpy() / 255.0
y_train = train_full.targets[:5000].numpy()
X_test  = test_set.data.reshape(-1, 784).numpy() / 255.0
y_test  = test_set.targets.numpy()

mlflow.sklearn.autolog(log_models=True)

with mlflow.start_run(run_name="sklearn_autolog") as run:
    clf = LogisticRegression(max_iter=20)
    clf.fit(X_train, y_train)
    print("test acc:", round(accuracy_score(y_test, clf.predict(X_test)), 4))

print("Автолог создал LoggedModel:", run.outputs.model_outputs[0].model_id)
print("Что залогировалось само: params, метрики, модель (Run detail page в UI)")

2026/11/03 17:50:05 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/11/03 17:50:05 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/11/03 17:50:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/11/03 17:50:11 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/11/03 17:50:11 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


test acc: 0.8068
🏃 View run sklearn_autolog at: http://127.0.0.1:5000/#/experiments/1/runs/2a05756408184b8bb6d44879b84d2c78
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Автолог создал LoggedModel: m-94a42dbed39746b59d61d9a3f9fcf502
Что залогировалось само: params, метрики, модель (Run detail page в UI)


## Системные показатели

С MLflow ≥ 2.8 клиент умеет логировать CPU/RAM/диск сам (GPU — при установленном `nvidia-ml-py`).
Мы включили это в начале секции, до первых запусков:
```python
os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"
os.environ["MLFLOW_SYSTEM_METRICS_SAMPLING_INTERVAL"] = "2"   # по умолчанию 10 с
```
Это фоновый поток на каждый run: он пишет `system/cpu_utilization_percentage`,
`system/system_memory_usage_megabytes`, `system/disk_*`, а при наличии GPU — `system/gpu_0_*`.
В UI запуска появится вкладка **System metrics**.


## Реестр моделей (Model Registry)

Версии моделей + **алиасы** (`@champion`, `@production` — в MLflow 3 они заменили устаревшие stages)
+ происхождение (какой run создал модель).

Из пяти запусков выберем лучший по `val/acc` и зарегистрируем именно его.


In [15]:
ours = {r["run_id"] for r in results}
best = mlflow.search_runs(order_by=["metrics.`val/acc` DESC"])
best = best[best["run_id"].isin(ours)].iloc[0]   # только запуски этой секции, без старых
best_uri = next(r["model_uri"] for r in results if r["run_id"] == best["run_id"])
print(f"Лучший запуск: {best['tags.mlflow.runName']}  val/acc={best['metrics.val/acc']:.3f}")

mv = mlflow.register_model(best_uri, "fashion_classifier") 

client = mlflow.MlflowClient()
client.set_registered_model_alias("fashion_classifier", "champion", mv.version)

loaded = mlflow.pyfunc.load_model("models:/fashion_classifier@champion")
print("Зарегистрировали версию", mv.version, "и загрузили по алиасу @champion")


Successfully registered model 'fashion_classifier'.
2026/11/03 17:52:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fashion_classifier, version 1


Лучший запуск: cnn_lr=0.001_h=128  val/acc=0.853
Зарегистрировали версию 1 и загрузили по алиасу @champion


Created version '1' of model 'fashion_classifier'.


## Артефакты и поиск лучшего запуска

Демонстрация: логируем **confusion matrix как артефакт** и находим лучший запуск **из кода**.

In [16]:
# Демонстрация: артефакт (confusion matrix) в MLflow + поиск лучшего запуска из кода
from sklearn.metrics import ConfusionMatrixDisplay

logger = MLflowLogger("confusion_matrix_demo")
model = train_model(CFG, logger=logger, verbose=False)

ys, preds = [], []
model.eval()
with torch.no_grad():
    for x, y in make_loaders(256)["val"]:
        preds.extend(model(x.to(DEVICE)).argmax(1).cpu().tolist())
        ys.extend(y.tolist())

fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(
    ys, preds, display_labels=CLASS_NAMES, ax=ax, xticks_rotation="vertical")
fig.savefig("cm.png", bbox_inches="tight")

# train_model уже вызвал logger.finish() -> run закрыт. Без этого log_artifact
# молча уедет в новый пустой run — поэтому возобновляем исходный по run_id.
with mlflow.start_run(run_id=logger.run_id):
    mlflow.log_artifact("cm.png")

runs_df = mlflow.search_runs(order_by=["metrics.`val/acc` DESC"])
ours = {r["run_id"] for r in results} | {logger.run_id}
best = runs_df[runs_df["run_id"].isin(ours)].iloc[0]
print("Лучший запуск:", best["tags.mlflow.runName"],
      "val/acc =", best["metrics.val/acc"])

2026/11/03 17:54:03 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/11/03 17:54:03 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/11/03 17:54:10 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/11/03 17:54:10 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run confusion_matrix_demo at: http://127.0.0.1:5000/#/experiments/1/runs/5e4e3aecc6e848a3aea32d08c7d95019
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/11/03 17:54:11 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/11/03 17:54:11 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/11/03 17:54:11 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/11/03 17:54:11 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run confusion_matrix_demo at: http://127.0.0.1:5000/#/experiments/1/runs/5e4e3aecc6e848a3aea32d08c7d95019
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Лучший запуск: cnn_lr=0.001_h=128 val/acc = 0.8525


<a id="sec3"></a>
# 4. Weights & Biases

SaaS-платформа (есть self-hosted версия). Главная сила — **удобный UI для команд**:
дашборды, сравнение запусков, артефакты с графом зависимостей, **sweeps**, отчёты.
Системные и GPU-метрики собираются **автоматически** (вкладка System у каждого запуска).

- Бесплатный тариф для личных проектов; для академических учреждений — бесплатная Pro-лицензия.
- Офлайн-режим: обучение без интернета, потом `wandb sync`.

In [23]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /home/defenser/.netrc.
wandb: Currently logged in as: defenser (defenser-) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


True

In [24]:
class WandbLogger:
    """Адаптер: интерфейс бейзлайна -> W&B."""

    def __init__(self, run_name, offline=False, project="seminar-tracking"):
        self.run = wandb.init(project=project, name=run_name,
                              mode="offline" if offline else "online")

    def log_params(self, cfg):
        self.run.config.update(cfg)

    def log_metric(self, name, value, step=0):
        self.run.log({name: value}, step=step)

    def log_artifact(self, path, type_="model"):
        art = wandb.Artifact(pathlib.Path(path).stem, type=type_)
        art.add_file(path)
        self.run.log_artifact(art)

    def finish(self):
        self.run.finish()


# Два запуска. Конфиг попадает в run.config -> UI рисует разрезы по параметрам.
for lr in (1e-4, 1e-3):
    logger = WandbLogger(f"cnn_lr={lr}")
    train_model({**CFG, "lr": lr}, logger=logger)

print("Откройте wandb.ai → project seminar-tracking.")
print("Вкладка System: CPU/RAM/GPU собраны БЕЗ единой строки кода мониторинга.")

эпоха 0: val_loss=1.6346  val_acc=0.568
эпоха 1: val_loss=0.9891  val_acc=0.704
эпоха 2: val_loss=0.8014  val_acc=0.723


train/loss,████▇▇▇▆▆▆▆▅▅▅▅▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▁▁▂▁▂▁▂▂▁▂
val/acc,▁▇█
val/loss,█▃▁
train/loss,0.79305
val/acc,0.7225
val/loss,0.80136


эпоха 0: val_loss=0.6747  val_acc=0.745
эпоха 1: val_loss=0.5281  val_acc=0.794
эпоха 2: val_loss=0.4796  val_acc=0.828


train/loss,█▅▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▂▂▂▂▂▁▂
val/acc,▁▅█
val/loss,█▃▁
train/loss,0.44484
val/acc,0.8275
val/loss,0.47957


Откройте wandb.ai → project seminar-tracking.
Вкладка System: CPU/RAM/GPU собраны БЕЗ единой строки кода мониторинга.


In [25]:
# Артефакты: сохраняем чекпоинт как версионируемый артефакт
pathlib.Path("checkpoints").mkdir(exist_ok=True)
torch.save(model.state_dict(), "checkpoints/fashion_cnn.pt")

with wandb.init(project="seminar-tracking", job_type="save-model",
                name="upload-checkpoint") as run:
    art = wandb.Artifact("fashion-cnn", type="model")
    art.add_file("checkpoints/fashion_cnn.pt")
    run.log_artifact(art)

# В UI: вкладка Artifacts — версия v0, граф зависимостей (lineage: кто и что использовал).
# Другой запуск может написать: run.use_artifact("fashion-cnn:latest") и скачать веса.

## Перебор гиперпараметров: Sweeps

Конфиг свипа описывает **как искать** (grid / random / **bayes**) и **что оптимизировать**.
Агент запускает прогоны с конфигами из свипа; UI показывает parallel coordinates и лучший запуск.
Есть ранний останов неудачных прогонов (hyperband: `early_terminate`).

In [26]:
sweep_config = {
    "method": "bayes",
    "metric": {"name": "val/acc", "goal": "maximize"},
    "parameters": {
        "lr":         {"distribution": "log_uniform_values", "min": 1e-5, "max": 1e-1},
        "dropout":    {"values": [0.1, 0.3, 0.5]},
        "batch_size": {"values": [64, 128]},
    },
}
sweep_id = wandb.sweep(sweep_config, project="seminar-tracking")


class AttachedLogger:
    """Адаптер для запуска, который уже создан агентом свипа."""

    def __init__(self, run):
        self.run = run

    def log_params(self, cfg):
        self.run.config.update(cfg, allow_val_change=True)

    def log_metric(self, name, value, step=0):
        self.run.log({name: value}, step=step)

    def finish(self):
        pass  # run закроется контекстным менеджером


def sweep_train(config=None):
    with wandb.init(config=config) as run:   # гиперпараметры приходят из свипа
        cfg = dict(run.config)
        cfg.setdefault("hidden_dim", 64)
        cfg["epochs"] = 1                    # для демо — по 1 эпохе
        train_model(cfg, logger=AttachedLogger(run))


wandb.agent(sweep_id, function=sweep_train, count=6)   # 6 прогонов для демо
# В UI: вкладка Sweeps → Parallel Coordinates: какие значения lr выжили?

Create sweep with ID: wvziehvj
Sweep URL: https://forge.coreweave.com/wandb/defenser-/seminar-tracking/sweeps/wvziehvj


wandb: Agent Starting Run: 9ybrvdg8 with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.3
wandb: 	lr: 0.001168515446012544
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /home/defenser/.netrc.


wandb: WARNING Config item 'batch_size' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'dropout' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'lr' was locked by 'sweep' (ignored update).


эпоха 0: val_loss=0.6504  val_acc=0.755


train/loss,██▇▅▅▄▄▄▄▃▄▄▃▃▂▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁
val/acc,▁
val/loss,▁
train/loss,0.69845
val/acc,0.755
val/loss,0.65041


wandb: Agent Starting Run: 0fpd0547 with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.1
wandb: 	lr: 1.297887861734896e-05
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /home/defenser/.netrc.


... (15 outputs left out)

wandb: Agent Starting Run: 412s8w2o with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.5
wandb: 	lr: 0.003370263182052256
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /home/defenser/.netrc.


wandb: WARNING Config item 'batch_size' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'dropout' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'lr' was locked by 'sweep' (ignored update).


эпоха 0: val_loss=0.5749  val_acc=0.779


train/loss,██▇▇▆▅▄▄▄▄▄▄▃▃▃▂▂▃▂▂▃▂▁▂▂▂▂▂▂▁▂▂▂▂▂▁▁▁▁▂
val/acc,▁
val/loss,▁
train/loss,0.7448
val/acc,0.779
val/loss,0.57489


## Отчёты (Reports) и офлайн-режим

- **Reports**: внутри проекта собираете отчёт с графиками и выводами — ссылка вместо «вот вам 5 скриншотов».
  Отлично для научрука/курсовой.
- **Офлайн-режим** — для кластеров без интернета и приватных данных:

In [27]:
# Офлайн-запуск: всё пишется локально в ./wandb/offline-run-*
logger = WandbLogger("offline_demo", offline=True)
train_model({**CFG, "epochs": 1}, logger=logger, verbose=False)

print("Данные лежат локально. Залить на сервер позже одним действием:")
print("  !wandb sync --sync-all")

train/loss,██▇▇▇▆▆▅▄▄▄▃▃▃▃▂▂▃▂▂▁▂▂▂▁▂▂▁▁▂▂▂▂▂▂▁▁▁▁▁
val/acc,▁
val/loss,▁
train/loss,0.67399
val/acc,0.7445
val/loss,0.67472


Данные лежат локально. Залить на сервер позже одним действием:
  !wandb sync --sync-all


<a id="sec4"></a>
# 5. Comet

Comet (comet.com) — SaaS-платформа экспериментального трекинга, ближайший конкурент W&B: эксперименты, метрики, параметры, код, автологирование, визуализации, артефакты, реестр моделей и датасетов, подбор гиперпараметров.

**Что здесь покажем:** адаптер к тому же `train_model`, два запуска, чекпоинт и confusion matrix — в **offline-режиме**, без аккаунта и интернета. Затем — что смотреть в UI и где Comet уступает.

Особенности:

- **визуализации из коробки**: ROC/PR, confusion matrix, распределения, графики кода — в UI, а не отдельными артефактами;
- **Opik** — отдельный open-source (Apache-2.0) проект Comet для наблюдаемости и оценки LLM/агентов: трейсы, оценки, датасеты, self-hosted. Классический MLOps-трекинг (Experiment Management) при этом продолжает работать как часть продукта;
- **offline-режим** (`OfflineExperiment`): всё пишется в ZIP, позже `comet upload` — удобно для кластера без сети;
- **self-hosted** — полностью on-prem только в enterprise-версии; Opik можно поднять самому.

Ограничения, о которых стоит сказать честно: меньше сообщества и готовых рецептов, чем у W&B/MLflow; SaaS-first (данные уходят в облако); полный on-prem — enterprise; для LLM-наблюдаемости нужен отдельный Opik, а не «всё в одном».


In [28]:
import pathlib, shutil, json
import comet_ml 

import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix

COMET_OFFLINE = False         
COMET_DIR = pathlib.Path("runs/comet")
shutil.rmtree(COMET_DIR, ignore_errors=True)
COMET_DIR.mkdir(parents=True, exist_ok=True)


class CometLogger:
    """Адаптер: интерфейс бейзлайна -> Comet."""

    def __init__(self, run_name, project="seminar-tracking", offline=COMET_OFFLINE):
        experiment_cls = comet_ml.OfflineExperiment if offline else comet_ml.Experiment
        kwargs = dict(
            project_name=project,
            auto_param_logging=False,    
            auto_metric_logging=False,
            log_code=True,             
            log_env_details=True,      
            log_git_metadata=False,     
            log_git_patch=False,
            auto_output_logging=None,
        )
        if offline:
            kwargs["offline_directory"] = str(COMET_DIR)
        self.exp = experiment_cls(**kwargs)
        self.exp.set_name(run_name)

    def log_params(self, cfg):
        self.exp.log_parameters(cfg)

    def log_metric(self, name, value, step=0):
        self.exp.log_metric(name, value, step=step)

    def log_artifact(self, path):
        self.exp.log_asset(str(path))

    def log_model(self, name, path):
        self.exp.log_model(name, str(path))

    def finish(self):
        pass

    def end(self):
        self.exp.end()

In [29]:
def log_confusion_matrix(logger, model, val_loader, title):
    """
    Логирует интерактивную confusion matrix Comet с примерами изображений
    в ячейках и сохраняет матрицу как confusion-matrix.json.
    """
    model.eval()
    y_true, y_pred = [], []
    all_images = []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            logits = model(x)
            preds = logits.argmax(1).cpu()

            y_pred.append(preds)
            y_true.append(y)

            # Батч нормализован (x - MEAN) / STD — возвращаем в [0, 1],
            # иначе после *255 и uint8 получится мусор
            x_vis = (x.cpu() * STD + MEAN).clamp(0, 1)

            # (B, C, H, W) -> (B, H, W, C), [0,1] -> [0,255] uint8
            imgs = (x_vis.permute(0, 2, 3, 1).numpy() * 255).round().astype(np.uint8)
            all_images.extend(imgs)

    y_true_cat = torch.cat(y_true).tolist()
    y_pred_cat = torch.cat(y_pred).tolist()

    logger.exp.log_confusion_matrix(
        y_true_cat,
        y_pred_cat,
        labels=CLASS_NAMES,
        title=title,
        images=all_images,
    )

    cm = confusion_matrix(y_true_cat, y_pred_cat, labels=range(len(CLASS_NAMES)))
    cm_data = {
        "labels": CLASS_NAMES,
        "matrix": cm.tolist(),
        "title": title,
    }
    json_path = pathlib.Path("confusion-matrix.json")
    with open(json_path, "w") as f:
        json.dump(cm_data, f, indent=2, ensure_ascii=False)
    logger.log_artifact(json_path)

    return cm

In [30]:
for lr in (1e-3, 1e-4):
    cfg = {**CFG, "lr": lr}
    logger = CometLogger(f"cnn_lr={lr}")
    model = train_model(cfg, logger=logger, verbose=False)

    ckpt = pathlib.Path("checkpoints") / f"comet_lr={lr}.pt"
    ckpt.parent.mkdir(exist_ok=True)
    torch.save(model.state_dict(), ckpt)

    logger.log_model("fashion-cnn", ckpt)

    # DataLoader для сбора изображений и предсказаний
    val_loader = DataLoader(val_set, batch_size=256, shuffle=False)
    log_confusion_matrix(logger, model, val_loader, title=f"lr={lr}")

    logger.end()   # ✅ закрываем эксперимент, чтобы всё ушло в Comet

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: mlflow, torch, tensorboard, sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/defenser1337/seminar-tracking/d857bf0f068a4b668587cab40a93215e

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : cnn_lr=0.001
COMET INFO:     url                   : https://www.comet.com/defenser1337/seminar-tracking/d857bf0f068a4b668587cab40a93215e
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     train/loss [186] : (0.3

In [31]:
archives = sorted(COMET_DIR.glob("*.zip"))
print("Офлайн-архивы:", [p.name for p in archives])
if archives:
    print("Загрузить позже: comet upload", archives[0])
else:
    print("Офлайн-архивов нет (COMET_OFFLINE=False или эксперименты уже отправлены).")

Офлайн-архивы: []
Офлайн-архивов нет (COMET_OFFLINE=False или эксперименты уже отправлены).


## Что смотреть в Comet

- **Metrics** — `train/loss` и `val/acc` по шагам; два запуска накладываются друг на друга.
- **Parameters** — `lr`, `batch_size`, `hidden_dim`, `dropout`: сравнение запусков по осям.
- **Assets / Artifacts** — чекпоинт `fashion-cnn` и `confusion-matrix.json`; матрица в UI открывается интерактивно, с примерами картинок по ячейкам.
- **Code / Environment** — если включить `log_code` и `log_env_details`, Comet приложит исходники и версии библиотек: запуск воспроизводится без «а у меня работало».
- **Optimizer** — подбор гиперпараметров (grid/random/Bayes); в offline-демо не запускаем, потому что контроллер живёт на сервере.

В отличие от MLflow и TensorBoard, confusion matrix и ROC/PR здесь **first-class**: их не надо рисовать matplotlib'ом и прикладывать картинкой.


## Ограничения Comet

- **Сообщество меньше**, чем у W&B и MLflow: меньше готовых рецептов и ответов на ошибки.
- **Цена.** Бесплатный тариф — для одного пользователя; для команды нужен платный план (актуальные условия — на comet.com/pricing). В отзывах пользователи отмечают, что с ростом использования продукт становится дорогим.
- **Полный self-hosted — только Enterprise.** Бесплатно и просто — облако; Opik можно поднять самому, но это LLM-часть, а не замена классическому трекингу.
- **SaaS-first.** Данные уходят в облако; для NDA-данных остаются Enterprise, on-prem или offline-режим с ручной выгрузкой.
- **Offline — это ZIP, а не сервер.** Пока не сделали `comet upload`, сравнить запуски в UI нельзя: в `runs/comet/` лежат только архивы.
- **Порядок импортов.** Автологирование требует импортировать `comet_ml` раньше `torch`. В нашем ноутбуке `torch` уже импортирован выше, поэтому Comet выдаст предупреждение; на результат это не влияет, потому что авто-логи отключены и мы логируем явно.
- **Большие файлы и много запусков.** В отзывах встречаются жалобы на ограничения при загрузке крупных файлов и на неудобство, когда экспериментов очень много; для своих чекпоинтов проверяйте на практике.
- **Интеграции.** Экосистема, по оценкам пользователей, уже, чем у W&B.

> Comet встраивает визуализации качества (confusion matrix, ROC/PR) прямо в логирование: интерактивная матрица с примерами изображений строится одним вызовом. У W&B аналогичные графики тоже есть (`wandb.plot`), но подключаются отдельно. Плата — меньшее сообщество, цена при росте команды и SaaS-first: полный self-hosted только в Enterprise, а для LLM-наблюдаемости, как и у W&B (Weave), нужен отдельный продукт — Opik.

<a id="sec5"></a>
# 6. Сравнение платформ

| Критерий | TensorBoard | MLflow | W&B | Comet |
|---|---|---|---|---|
| Тип | UI + файл-логи | open-source платформа | SaaS + self-hosted | SaaS-first (Opik — OSS) |
| Самостоятельный хостинг | сам по себе локальный | ✅ тривиально (SQLite/Postgres + диск/S3) | ✅ Docker (`wandb/server`) | on-prem только в Enterprise; Opik ✅ |
| Скаляры / гистограммы / изображения | ✅ | ⚠️ скаляры и картинки; гистограмм нет | ✅ | ✅ + аудио, текст, 3D, таблицы |
| Автологирование | ❌ (вручную) | ✅ очень широкое | ✅ интеграции | ✅ PyTorch, Lightning, Keras, sklearn (`import comet_ml` раньше `torch`) |
| Свипы гиперпараметров | ❌ (есть вкладка HParams для просмотра) | частично (Optuna/Hyperopt + код) | ✅ grid/random/bayes + early stop | ✅ Comet Optimizer: grid/random/bayes, параллельные агенты |
| Артефакты + версионирование | ❌ | ✅ | ✅ (+ lineage) | ✅ Artifacts и assets, версии |
| Реестр моделей | ❌ | ✅ версии + алиасы (`@champion`) | ✅ Registry | ✅ версии, стадии (dev → staging → prod), вебхуки для CI/CD |
| Системные / GPU-метрики | ❌ (вручную) | ✅ (MLflow ≥ 2.8) | ✅ из коробки | ✅ вкладка System: CPU / GPU / RAM |
| Код и окружение | ❌ | git-коммит как тег; зависимости с моделью | git-коммит, код (опционально) | ✅ исходники, git-патч, версии библиотек (`log_code`, `log_env_details`) |
| Сравнение запусков | базовое | ✅ таблица / parallel coords | ✅ богатое | ✅ таблица, панели, diff кода и параметров |
| Готовые графики качества | ❌ | через артефакт (matplotlib) | через `wandb.plot` | ✅ confusion matrix, ROC/PR из коробки |
| Совместная работа / отчёты | ❌ (tensorboard.dev закрыт) | UI на общем сервере | ✅ Reports | ✅ Workspaces, Panels (в т. ч. свои на Python), Reports |
| Офлайн | локально по умолчанию | локально по умолчанию | ✅ `WANDB_MODE=offline` + `wandb sync` | ✅ `OfflineExperiment` → ZIP + `comet upload` |
| LLM / агенты | ❌ | ✅ MLflow Tracing (GenAI) | ✅ Weave | ✅ Opik (OSS, отдельный проект) |
| Лицензия и цена | OSS, бесплатно | OSS (Apache 2.0), бесплатно | free для личного/академического; платные тарифы | free-тариф (1 пользователь), платный Team, Enterprise |
| Приватность | ✅ всё локально | ✅ всё локально | облако / self-hosted | облако; on-prem только Enterprise; offline с ручной выгрузкой |
| Главная фишка | быстро заглянуть внутрь запуска | своё хранилище + реестр | UI и свипы для команд | визуализации из коробки + код/окружение запуска |

Другие игроки, о которых стоит знать: Neptune.ai, ClearML, Aim, Polyaxon, DVC (версионирование данных).

<a id="sec6"></a>
# 7. Заключение

Четыре мысли, которые стоит унести с семинара:

- **Инструмент выбирают под задачу и ограничения**, а не по хайпу. TensorBoard — быстро заглянуть внутрь запуска; MLflow — своя «картотека» запусков и реестр моделей, всё остаётся у вас; W&B и Comet — готовый командный UI, отчёты и свипы. Главные критерии: нужен ли общий доступ для команды, можно ли отправлять данные в облако, нужен ли реестр моделей.
- **Логировать надо по шагам и вместе с контекстом**: гиперпараметры, seed, версия кода и данных, окружение. Итоговое число без этого не объясняет, как вы к нему пришли, и его нельзя воспроизвести.
- **Результат — это артефакт.** Метрика без чекпоинта, конфига и графика не проверяется и не передаётся коллегам.
- **ИИ-агенты упрощают подключение трекинга, но не заменяют дисциплину.** Агент может быстро добавить логирование и разобрать историю запусков, но что логировать и как называть запуски, решает человек. Чем больше экспериментов запускается автоматически, тем нужнее общий журнал. Проверяйте, что агент не логирует лишнего и не отправляет ключи и приватные данные во внешний сервис.

### Чек-лист перед запуском эксперимента
- [ ] Зафиксированы seed и гиперпараметры (конфиг сохраняется, а не только лежит в коде)
- [ ] Метрики пишутся по шагам, train и val на одной оси
- [ ] Сохранены версия кода (git-коммит) и окружение (версии библиотек)
- [ ] Указана версия или хэш данных
- [ ] Лучший чекпоинт и ключевые графики сохранены как артефакты
- [ ] Запуски названы осмысленно, по ним можно найти нужный
- [ ] Проверено, что данные под NDA не уходят в облако, а ключи API не попадают в логи

### Кейсы для дискуссии

Выберите платформу и обоснуйте:
1. Учебный проект в Kaggle-ноутбуке, данных ноль приватных.
2. Медицинские данные под NDA на закрытом кластере **без интернета**.
3. Команда из 10 человек, нужен реестр моделей и отчёты для менеджера.

<details><summary>🤔 Возможные аргументы</summary>

1. **TensorBoard или MLflow**: без аккаунтов, минимум настройки; MLflow даст реестр и сравнение из кода «на вырост».
2. **MLflow self-hosted** (метрики в своём Postgres, артефакты в своём S3) или **W&B offline + sync** в DMZ; W&B self-hosted, если важны свипы/UI.
3. **W&B**: Reports + свипы + реестр; либо **MLflow на общем сервере**, если всё должно остаться на своём железе.
</details>

<a id="appendix"></a>
# 8. Источники и ссылки

**TensorBoard**
- Доки: https://www.tensorflow.org/tensorboard/get_started
- HParams (подбор гиперпараметров): https://www.tensorflow.org/tensorboard/hyperparameter_tuning_with_hparams
- `SummaryWriter` (PyTorch API): https://pytorch.org/docs/stable/tensorboard.html
- Туториал с проектором: https://pytorch.org/tutorials/intermediate/tensorboard_tutorial.html
- Профайлер: https://pytorch.org/tutorials/intermediate/tensorboard_profiler_tutorial.html

**MLflow**
- Quickstart: https://mlflow.org/docs/latest/getting-started/
- Сервер (self-hosting): https://mlflow.org/docs/latest/tracking/server.html
- Системные метрики: https://mlflow.org/docs/latest/system-metrics/
- Реестр моделей: https://mlflow.org/docs/latest/model-registry/
- Что нового в MLflow 3 (в основном про GenAI и трейсинг): https://mlflow.org/blog/mlflow-3-launch

**Weights & Biases** (с 2025 года часть CoreWeave; документация и цены переехали)
- Quickstart: https://docs.wandb.ai/quickstart
- Sweeps: https://docs.wandb.ai/guides/sweeps
- Артефакты: https://docs.wandb.ai/guides/artifacts
- Отчёты: https://docs.wandb.ai/guides/reports
- Офлайн + sync: https://docs.wandb.ai/support/models/articles/how-do-i-run-wandb-offline
- Self-hosted (образ `wandb/local`, для продакшена нужна лицензия): https://github.com/wandb/server
- Тарифы (включая бесплатную академическую лицензию): https://wandb.ai/site/pricing
- Бесплатные курсы: https://www.wandb.courses/

**Comet**
- Доки: https://www.comet.com/docs/ · Opik (OSS): https://github.com/comet-ml/opik
- Тарифы и self-hosted: https://www.comet.com/site/pricing/

**Общее**
- Обзор инструментов трекинга (2024, часть информации устарела; Neptune закрыт в 2026): https://neptune.ai/blog/best-ml-experiment-tracking-tools
- Full Stack Deep Learning, Lab 4: Experiment Management: https://fullstackdeeplearning.com/course/2022/lab-4-experiment-management/
- Made With ML: https://madewithml.com/
- Иллюстрации (рис. 1–3): ChatGPT (OpenAI), Claude (Anthropic)